In [0]:
-- ============================================================
-- GOLD: dim_condition
-- Classifies SNOMED conditions by TYPE (disorder vs finding/
-- situation) and CATEGORY. CKD staging gated on renal terms.
-- Category regexes tuned against actual condition frequencies.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.dim_condition AS
WITH distinct_conditions AS (
  SELECT DISTINCT condition_code, condition_desc
  FROM health_lakehouse.silver.conditions
),
typed AS (
  SELECT
    condition_code,
    condition_desc,
    CASE
      WHEN condition_desc LIKE '%(disorder)%' THEN 'disorder'
      WHEN condition_desc LIKE '%(finding)%' THEN 'finding'
      WHEN condition_desc LIKE '%(situation)%' THEN 'situation'
      WHEN condition_desc LIKE '%(morphologic abnormality)%' THEN 'morphologic'
      WHEN LOWER(condition_desc) RLIKE 'retinopathy|due to type ii diabetes' THEN 'disorder'
      ELSE 'unclassified'
    END AS condition_type,
    CASE
      -- renal/urinary first (diabetic kidney disease is renal, not metabolic)
      WHEN LOWER(condition_desc) RLIKE 'kidney|renal|nephro|dialysis|microalbuminuria|proteinuria|cystitis|urinary tract'
        THEN 'renal_urinary'
      WHEN LOWER(condition_desc) RLIKE 'diabet|prediabetes|metabolic syndrome|hyperlipidemia|hypertriglycerid|hyperglycemia|cholesterol'
        THEN 'metabolic'
      WHEN LOWER(condition_desc) RLIKE 'hypertension|ischemic heart|coronary|cardiac|myocardial|heart failure|atrial'
        THEN 'cardiovascular'
      WHEN LOWER(condition_desc) RLIKE 'asthma|bronchitis|sinusitis|pharyngitis|pneumonia|copd|respiratory|apnea|emphysema|rhinitis'
        THEN 'respiratory'
      WHEN LOWER(condition_desc) RLIKE 'migraine|seizure|epilepsy|alzheimer|neuropath|parkinson|dementia'
        THEN 'neurological'
      WHEN LOWER(condition_desc) RLIKE 'anxiety|depress|stress|psychiat|bipolar|schizo|sleep disorder|attention deficit'
        THEN 'mental_health'
      WHEN LOWER(condition_desc) RLIKE 'otitis|streptococcal|sore throat'
        THEN 'ent_infection'
      WHEN LOWER(condition_desc) RLIKE 'gingiv|dental|caries|tooth|teeth|filling|molar|alveolitis'
        THEN 'dental'
      WHEN LOWER(condition_desc) RLIKE 'fracture|sprain|laceration|injury|burn|wound|concussion'
        THEN 'injury'
      WHEN LOWER(condition_desc) RLIKE 'pregnan|miscarriage|tubal|labor|eclampsia|blighted ovum'
        THEN 'obstetric'
      WHEN LOWER(condition_desc) RLIKE 'cancer|neoplasm|carcinoma|tumor|malignan|polyp'
        THEN 'oncology'
      WHEN LOWER(condition_desc) RLIKE 'anemia|leukemia|hematolog'
        THEN 'haematology'
      WHEN LOWER(condition_desc) RLIKE 'pain|arthritis|osteo|back|neck|scoliosis|fibromyalgia'
        THEN 'musculoskeletal'
      WHEN LOWER(condition_desc) RLIKE 'employ|education|housing|income|criminal|social|isolation|violence|abuse|alcohol|labor force|refugee'
        THEN 'social_determinant'
      WHEN LOWER(condition_desc) RLIKE 'medication review'
        THEN 'administrative'
      ELSE 'other'
    END AS condition_category,
    CASE
      WHEN LOWER(condition_desc) RLIKE 'kidney|renal' AND condition_desc LIKE '%stage 1%' THEN 1
      WHEN LOWER(condition_desc) RLIKE 'kidney|renal' AND condition_desc LIKE '%stage 2%' THEN 2
      WHEN LOWER(condition_desc) RLIKE 'kidney|renal' AND condition_desc LIKE '%stage 3%' THEN 3
      WHEN LOWER(condition_desc) RLIKE 'kidney|renal' AND condition_desc LIKE '%stage 4%' THEN 4
      WHEN LOWER(condition_desc) LIKE '%end-stage renal%' THEN 5
    END AS ckd_stage
  FROM distinct_conditions
)
SELECT * FROM typed;

In [0]:
SELECT condition_category, COUNT(*) AS distinct_codes
FROM health_lakehouse.gold.dim_condition
WHERE condition_type = 'disorder'
GROUP BY 1 ORDER BY distinct_codes DESC;

In [0]:
-- verifying CKD staging
SELECT ckd_stage, condition_desc
FROM health_lakehouse.gold.dim_condition
WHERE ckd_stage IS NOT NULL
ORDER BY ckd_stage;

In [0]:
-- verifying conditions in the 'Other' category
SELECT c.condition_desc, cnt.n
FROM health_lakehouse.gold.dim_condition c
JOIN (SELECT condition_code, COUNT(*) n FROM health_lakehouse.silver.conditions GROUP BY condition_code) cnt
  ON c.condition_code = cnt.condition_code
WHERE c.condition_type = 'disorder' AND c.condition_category = 'other'
ORDER BY cnt.n DESC LIMIT 5;

In [0]:
-- ============================================================
-- GOLD: dim_patient
-- Conformed patient dimension
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.dim_patient AS
SELECT
  patient_id,
  birth_date,
  death_date,
  is_deceased,
  age,
  age_band,
  gender,
  race,
  ethnicity,
  marital_status,
  city,
  state,
  county,
  zip,
  income,
  healthcare_expenses,
  healthcare_coverage
FROM health_lakehouse.silver.patients;

In [0]:
-- ============================================================
-- GOLD: dim_provider
-- Provider denormalized with its organization.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.dim_provider AS
SELECT
  pr.Id AS provider_id,
  pr.NAME AS provider_name,
  pr.SPECIALITY AS speciality,
  pr.GENDER AS provider_gender,
  o.Id AS organization_id,
  o.NAME AS organization_name,
  o.CITY AS organization_city,
  o.STATE AS organization_state
FROM health_lakehouse.bronze.providers pr
LEFT JOIN health_lakehouse.bronze.organizations o
  ON pr.ORGANIZATION = o.Id;

In [0]:
-- ============================================================
-- GOLD: dim_payer
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.dim_payer AS
SELECT
  Id AS payer_id,
  NAME AS payer_name,
  OWNERSHIP AS ownership
FROM health_lakehouse.bronze.payers;

In [0]:
-- ============================================================
-- GOLD: dim_date
-- Generated date spine covering the data's time range.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.dim_date AS
WITH bounds AS (
  SELECT
    DATE(MIN(start_ts)) AS min_d,
    DATE(MAX(start_ts)) AS max_d
  FROM health_lakehouse.silver.encounters
),
date_spine AS (
  SELECT explode(sequence(min_d, max_d, INTERVAL 1 DAY)) AS date_day
  FROM bounds
)
SELECT
  date_day,
  YEAR(date_day) AS year,
  QUARTER(date_day) AS quarter,
  MONTH(date_day) AS month,
  DATE_FORMAT(date_day, 'MMMM')AS month_name,
  DAY(date_day) AS day_of_month,
  DAYOFWEEK(date_day) AS day_of_week,
  DATE_FORMAT(date_day, 'EEEE') AS day_name,
  WEEKOFYEAR(date_day) AS week_of_year,
  CASE WHEN DAYOFWEEK(date_day) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend
FROM date_spine;

In [0]:
-- verifying total rows per generated gold table
SELECT 'dim_patient'   AS dim, COUNT(*) AS rows FROM health_lakehouse.gold.dim_patient
UNION ALL SELECT 'dim_provider',  COUNT(*) FROM health_lakehouse.gold.dim_provider
UNION ALL SELECT 'dim_payer',     COUNT(*) FROM health_lakehouse.gold.dim_payer
UNION ALL SELECT 'dim_condition', COUNT(*) FROM health_lakehouse.gold.dim_condition
UNION ALL SELECT 'dim_date',      COUNT(*) FROM health_lakehouse.gold.dim_date;

##  FACT TABLES

In [0]:
-- ============================================================
-- GOLD: fact_encounters
-- Grain: one row per encounter
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.fact_encounters AS
SELECT
  e.encounter_id,
  e.patient_id,
  e.provider_id,
  e.payer_id,
  DATE(e.start_ts) AS encounter_date,
  e.start_ts,
  e.stop_ts,
  e.encounter_class,
  e.encounter_code,
  e.encounter_desc,
  e.los_hours,
  e.base_cost,
  e.total_claim_cost,
  e.payer_coverage,
  (e.total_claim_cost - e.payer_coverage) AS patient_liability
FROM health_lakehouse.silver.encounters e;

In [0]:
-- ============================================================
-- GOLD: fact_conditions
-- Grain: one row per patient-condition episode.
-- Joins dim_condition to carry type/category/ckd_stage onto the fact.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.fact_conditions AS
SELECT
  c.patient_id,
  c.encounter_id,
  c.condition_code,
  d.condition_desc,
  d.condition_type,
  d.condition_category,
  d.ckd_stage,
  c.onset_date,
  c.resolved_date,
  c.is_active,
  c.duration_days
FROM health_lakehouse.silver.conditions c
LEFT JOIN health_lakehouse.gold.dim_condition d
  ON c.condition_code = d.condition_code;

In [0]:
-- ============================================================
-- GOLD: fact_medications
-- Grain: one row per prescription.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.fact_medications AS
SELECT
  m.patient_id,
  m.encounter_id,
  m.payer_id,
  m.medication_code,
  m.medication_desc,
  DATE(m.start_ts) AS start_date,
  m.start_ts,
  m.stop_ts,
  m.duration_days,
  m.is_ongoing,
  m.dispenses,
  m.base_cost,
  m.total_cost,
  m.payer_coverage,
  m.reason_code,
  m.reason_desc
FROM health_lakehouse.silver.medications m;

In [0]:
-- ============================================================
-- GOLD: fact_observations
-- Grain: one plausible numeric measurement (curated panel only).
-- Built on the validated vitals/labs table.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.fact_observations AS
SELECT
  o.patient_id,
  o.encounter_id,
  DATE(o.observed_ts) AS observation_date,
  o.observed_ts,
  o.measure_group,
  o.observation_code,
  o.observation_desc,
  o.value_numeric,
  o.units
FROM health_lakehouse.silver.observations_vitals_labs o
WHERE o.is_plausible = TRUE; -- exclude the flagged implausible values

In [0]:
-- Verifying total rows per generated fact table
SELECT 'fact_encounters'   AS fact, COUNT(*) AS rows FROM health_lakehouse.gold.fact_encounters
UNION ALL SELECT 'fact_conditions',   COUNT(*) FROM health_lakehouse.gold.fact_conditions
UNION ALL SELECT 'fact_medications',  COUNT(*) FROM health_lakehouse.gold.fact_medications
UNION ALL SELECT 'fact_observations', COUNT(*) FROM health_lakehouse.gold.fact_observations;